# 🔴 Solution: Implement Softmax（参考版）

## 📋 题目描述

**难度：** Easy

实现 softmax 函数。

Softmax 通过指数化和归一化将原始 logits 转换为概率分布，用于分类和注意力机制。

**签名:** `my_softmax(x, dim=-1) -> Tensor`

**参数:**
- `x` — 任意形状的输入张量
- `dim` — 计算 softmax 的维度

**返回:** 概率张量（沿 dim 求和为 1），形状与输入相同

**约束:**
- 在 exp 之前减去最大值以保证数值稳定
- 必须处理大值而不产生 NaN/Inf

**提示：** 1. `x_max = x.max(dim=dim, keepdim=True).values`
2. `e_x = exp(x - x_max)`
3. `return e_x / e_x.sum(dim=dim, keepdim=True)`


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    # ---- Step 1: 数值稳定性——减去最大值 ----
    # softmax(x_i) = exp(x_i) / Σ exp(x_j)
    # 问题：如果 x 中有很大的正数（如 1000），exp(1000) 会溢出 inf
    # 技巧：减去最大值 max(x)，因为 softmax 对平移不变：
    #   softmax(x_i - c) = exp(x_i - c) / Σ exp(x_j - c)
    #                     = exp(x_i)·exp(-c) / Σ exp(x_j)·exp(-c)
    #                     = softmax(x_i)
    # 这样最大值变为 0，exp(0)=1，其他值为负数，exp 一定在 (0,1] 范围
    x_max = x.max(dim=dim, keepdim=True).values

    # ---- Step 2: 减去最大值后的指数 ----
    # x_shifted 的最大值为 0，所有 exp 结果 ∈ (0, 1]，不会溢出
    x_shifted = x - x_max

    # ---- Step 3: 计算指数 ----
    exp_x = torch.exp(x_shifted)

    # ---- Step 4: 求和（归一化分母） ----
    # keepdim=True 保持维度，使后续除法能正确广播
    # 例如 x shape=[3,4]，dim=-1 → sum shape=[3,1]（而非 [3]）
    sum_exp = exp_x.sum(dim=dim, keepdim=True)

    # ---- Step 5: 逐元素除法得到概率分布 ----
    # 结果每一行（沿 dim）之和为 1，值域 (0, 1]
    return exp_x / sum_exp

In [ ]:
# Verify
x = torch.tensor([1.0, 2.0, 3.0])
print("Output:", my_softmax(x, dim=-1))
print("Sum:   ", my_softmax(x, dim=-1).sum())
print("Ref:   ", torch.softmax(x, dim=-1))

In [ ]:
# Run judge
from torch_judge import check
check("softmax")

## 📝 核心思路总结

1. **数值稳定性**：减去 max 防止 exp 溢出，这是 softmax 实现的核心技巧
2. **keepdim 的作用**：保持求和维度，使除法能正确广播
3. **平移不变性**：softmax(x - c) = softmax(x)，数学上等价
4. **面试要点**：温度参数、梯度推导、与 CrossEntropyLoss 的关系